### Replace class_embed with a HypLL-based hyperbolic classification head

In [ ]:
import math
import torch
import torch.nn as nn
from hypll.manifolds.poincare_ball import Curvature, PoincareBall
from hypll.manifolds.poincare_ball.math.diffgeom import expmap0, project
from hypll.manifolds.poincare_ball.math.linalg import poincare_hyperplane_dists


class HyperbolicDETRHead(nn.Module):
    """
    Hyperbolic classification head for RF-DETR using HypLL's Poincare ball.

    Replaces nn.Linear class_embed with HNN++ hyperbolic hyperplane classification.
    Features are mapped to the Poincare ball via expmap, then classified using
    signed geodesic distances to learnable hyperbolic hyperplanes.

    Well-suited for hierarchical vehicle classes:
      Passenger (Sedan, SUV, Pickup) / Commercial (Van, Truck, Bus) / Motorcycle
    """

    def __init__(self, input_dim, num_classes, curvature=0.1):
        super().__init__()
        self.input_dim = input_dim
        self.num_classes = num_classes

        # HypLL Poincare ball manifold (fixed curvature — no RiemannianAdam needed)
        self.manifold = PoincareBall(c=Curvature(value=curvature, requires_grad=False))

        # Euclidean projection to prepare features for hyperbolic mapping
        self.proj = nn.Linear(input_dim, input_dim)

        # Hyperplane orientation vectors z_k for each class
        # Shape: [input_dim, num_classes], manifold dim = 0
        self.z = nn.Parameter(torch.empty(input_dim, num_classes))
        nn.init.normal_(self.z, mean=0, std=(2 * input_dim * num_classes) ** -0.5)

        # Hyperplane offsets in hyperbolic space
        self.r = nn.Parameter(torch.zeros(num_classes))

        # Euclidean bias added to logits (for focal loss initialization)
        self.bias = nn.Parameter(torch.zeros(num_classes))

    def forward(self, x):
        """
        Args:
            x: [..., input_dim]  e.g. [dec_layers, batch, queries, 256]
        Returns:
            logits: [..., num_classes]  compatible with sigmoid focal loss
        """
        leading_shape = x.shape[:-1]
        D = x.shape[-1]

        x_flat = x.reshape(-1, D)           # [N, D]

        x_proj = self.proj(x_flat)           # [N, D]

        c = self.manifold.c()                # curvature scalar tensor

        # Map to Poincare ball via exponential map at origin
        x_ball = expmap0(x_proj, c, dim=-1)  # [N, D], guaranteed on ball

        # HNN++ classification: signed distances to hyperbolic hyperplanes
        hyp_logits = poincare_hyperplane_dists(
            x_ball, self.z, self.r, c, dim=-1
        )                                    # [N, num_classes]

        # Add Euclidean bias (initialized for focal loss)
        logits = hyp_logits + self.bias

        return logits.reshape(*leading_shape, self.num_classes)

###  Monkey-patch reinitialize_detection_head

In [ ]:
import types

def _hyp_reinit_detection_head(self, num_classes):
    """Override: create hyperbolic head instead of nn.Linear."""
    del self.class_embed
    hyp_head = HyperbolicDETRHead(
        input_dim=self.transformer.d_model,
        num_classes=num_classes,
        curvature=0.1,
    )
    self.add_module("class_embed", hyp_head)

    # Focal loss bias init (same as original RF-DETR)
    prior_prob = 0.01
    bias_value = -math.log((1 - prior_prob) / prior_prob)
    with torch.no_grad():
        self.class_embed.bias.data.fill_(bias_value)

    # Handle two-stage if applicable
    if self.two_stage:
        import copy
        del self.transformer.enc_out_class_embed
        self.transformer.add_module(
            "enc_out_class_embed",
            nn.ModuleList(
                [copy.deepcopy(self.class_embed) for _ in range(self.group_detr)]
            ),
        )
    print(f"Hyperbolic head initialized: {num_classes} classes, dim={self.transformer.d_model}")

### Load dataset

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="sjKAVuO8Lkaq5h2dDfDA")
project = rf.workspace("fyp-vfrgn").project("veiculos-contar-dnosk")
version = project.version(3)
dataset = version.download("coco")

### Instantiate the model

In [ ]:
from rfdetr import RFDETRBase

model = RFDETRBase(pretrain_weights="rf-detr-base.pth")

# Apply to the LWDETR model inside RFDETRBase instance
model.model.model.reinitialize_detection_head = types.MethodType(
    _hyp_reinit_detection_head, model.model.model
)

### Train

In [ ]:
history = []

def callback2(data):
	history.append(data)

model.callbacks["on_fit_epoch_end"].append(callback2)

model.train(
    dataset_dir=dataset.location,
    epochs=30,
    batch_size=4,
    grad_accum_steps=4,
    lr=1e-4,
    lr_encoder=1.5e-4,
    weight_decay=1e-4,
    amp=True,
)